<b>Executing GLiNER models</b>

This notebook documents how to generate NER predictions aligned with the ArchOnto ontology from datasets available at: [https://doi.org/10.6084/m9.figshare.28970633](https://doi.org/10.6084/m9.figshare.28970633).

<b>Import the required libraries</b>

In [ ]:
from pathlib import Path
from tqdm import tqdm
from gliner import GLiNER
import os
import pandas as pd
import re
import spacy
import time
import sys
import json
import csv

<b>Define and parse mapping rules between entity descriptors and ArchOnto classes</b>

In [ ]:
# Dictionary with labels mapping
label_mapping={"Person":"E21", "Organization":"E74", "Place":"E53", "Title": "ARE2", "Role": "ARE8", "Event": "E5", "Date": "E52", "Dimension": "E54"}
labels = labels_mapping.keys()

# OR just define a list of labels
#labels=["Person", "Organization", "Place", "Title", "Role", "Event", "Date", "Dimension"]

print(f"Labels: {labels}")
print(f"Mapping: {label_mapping}")

<b>Load ground-truth files with annotated entities in CoNLL-03 format</b>

In [ ]:
def conll_to_text(conll_path):
    sentences = []
    current_sent = []
    
    with open(conll_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sent:
                    sentences.append(' '.join(current_sent))
                    current_sent = []
            else:
                parts = line.split("\t")
                if parts:
                    current_sent.append(parts[0])
    
    return sentences

print("Loading input data...")

# Dataset options https://doi.org/10.6084/m9.figshare.28970633
text = conll_to_text("ner/datasets/ner-domain-gen/test.tsv")

<b>Create dataframe of predictions in BIO CoNLL-03 format</b>

In [ ]:
# Portuguese spaCy model options https://spacy.io/models/pt
nlp = spacy.load("pt_core_news_sm")

def convert_to_conll(text, entities, df):
    ent_lookup = {}
    for ent in sorted(entities, key=lambda e: e["start"]):
        for i in range(ent["start"], ent["end"]):
            ent_lookup[i] = ent

    doc = nlp(text)
    tokens_list = []
    offset = 0

    for token in doc:
        start = text.find(token.text, offset)
        if start == -1:
            break
        end = start + len(token.text)
        tokens_list.append({"text": token.text, "start": start, "end": end})
        offset = end
    
    tokens_text = []
    preds = []
    prev_ent = None
    ner=[]
    
    for tok in tokens_list:
        ent = ent_lookup.get(tok["start"])

        if ent is None:
            label = "O"
            prev_ent = None
        else:
            if ent != prev_ent:
                prefix = "B"
                prev_ent = ent
            else:
                prefix = "I"
            label = f"{prefix}-{label_mapping[ent['label']]}"
        ner.append((tok['text'], label))
    ner.append((),)

    if df.empty:
        return pd.DataFrame(ner)
    else:
        return pd.concat([df, pd.DataFrame(ner)], ignore_index=True)

<b>Load and execute GLiNER model</b>

In [ ]:
# Gliner model options https://huggingface.co/urchade 
gliner_model="urchade/gliner_small-v2.1"

print(f"Loading GLiNER model: {gliner_model}")
model = GLiNER.from_pretrained(gliner_model)

with tqdm(total=len(text), desc="Inference", unit="sent") as pbar:
    for index in range(len(text)):
        entities = model.predict_entities(text[index], labels, threshold=0.5)
        df=convert_to_conll(text[index], entities, df)
        pbar.update(1)

<b>Print execution inference time</b>

In [ ]:
print("--- %s seconds ---" % (pbar.format_dict["elapsed"]))
pbar.close()

<b>Save prediciton file</b>

In [ ]:
dst_path="path/to/your/output/test.tsv"
print(f"Saving annotated file in {dst_path}")
df.to_csv(dst_path, sep='\t',index=False, header=False, lineterminator='\n', quoting=csv.QUOTE_NONE, quotechar='"')